## Soil Data for the corn belt

[https://www.nrcs.usda.gov/resources/data-and-reports/soil-survey-geographic-database-ssurgo](https://www.nrcs.usda.gov/resources/data-and-reports/soil-survey-geographic-database-ssurgo)

There is a SQL database to query from called [Soil Data Access Query](https://sdmdataaccess.nrcs.usda.gov/Query.aspx). This might be a way to get tabular data to then combine with the soil and weather data. 

## Import Libraries

In [1]:
import pandas as pd
import numpy as np
import requests
import os
import time

## Load Yield/Weather Data

In [2]:
yield_weather_df = pd.read_csv('../data/raw/corn_belt_yield_weather.csv', dtype={"fips": str})
yield_weather_df.head()

,state_name,state_alpha,state_ansi,county_ansi,fips,county_name,year,short_desc,unit_desc,statisticcat_desc,...,vp_mean_pa_m03,vp_mean_pa_m04,vp_mean_pa_m05,vp_mean_pa_m06,vp_mean_pa_m07,vp_mean_pa_m08,vp_mean_pa_m09,vp_mean_pa_m10,vp_mean_pa_m11,vp_mean_pa_m12
0,ILLINOIS,IL,17,11,17011,BUREAU,2025,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,...,669.014194,860.098667,1149.159032,1970.153333,2279.072258,1885.761613,1442.058333,1146.211935,628.067667,357.549355
1,ILLINOIS,IL,17,11,17011,BUREAU,2024,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,...,652.787419,916.069000,1400.653548,1984.589000,1915.040968,1906.293871,1466.279667,978.475806,777.935667,462.975000
2,ILLINOIS,IL,17,11,17011,BUREAU,2023,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,...,551.773226,786.947000,1208.501613,1535.258667,1945.164194,1912.132581,1583.150333,1067.424516,567.281000,614.017097
3,ILLINOIS,IL,17,11,17011,BUREAU,2022,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,...,561.575484,759.192000,1471.162581,1827.592333,2026.516452,1920.243871,1510.500333,831.142258,627.178667,393.786774
4,ILLINOIS,IL,17,11,17011,BUREAU,2021,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,...,594.198387,847.325333,1287.721935,2010.488000,2090.430323,2056.888065,1548.647000,1280.342903,565.100000,504.136452


## Checking Data

In [3]:
yield_weather_df.shape

(24644, 97)

In [4]:
yield_weather_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24644 entries, 0 to 24643
Data columns (total 97 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   state_name                 24644 non-null  object 
 1   state_alpha                24644 non-null  object 
 2   state_ansi                 24644 non-null  int64  
 3   county_ansi                24644 non-null  int64  
 4   fips                       24644 non-null  object 
 5   county_name                24644 non-null  object 
 6   year                       24644 non-null  int64  
 7   short_desc                 24644 non-null  object 
 8   unit_desc                  24644 non-null  object 
 9   statisticcat_desc          24644 non-null  object 
 10  Value                      24644 non-null  float64
 11  latitude                   24644 non-null  float64
 12  longitude                  24644 non-null  float64
 13  prcp_total_mm_m01          24644 non-null  flo

In [5]:
yield_weather_df.describe()

,state_ansi,county_ansi,year,Value,latitude,longitude,prcp_total_mm_m01,prcp_total_mm_m02,prcp_total_mm_m03,prcp_total_mm_m04,...,vp_mean_pa_m03,vp_mean_pa_m04,vp_mean_pa_m05,vp_mean_pa_m06,vp_mean_pa_m07,vp_mean_pa_m08,vp_mean_pa_m09,vp_mean_pa_m10,vp_mean_pa_m11,vp_mean_pa_m12
count,24644.000000,24644.000000,24644.000000,24644.000000,24644.000000,24644.000000,24644.000000,24644.000000,24644.000000,24644.000000,...,24644.000000,24644.000000,24644.000000,24644.000000,24644.000000,24644.000000,24644.000000,24644.000000,24644.000000,24644.000000
mean,28.079127,93.939012,2011.815533,147.829005,41.299657,-91.543901,46.268875,48.823535,68.422388,95.190592,...,522.005559,757.236564,1199.100607,1731.095781,2000.412277,1874.845454,1438.836336,933.330077,601.022181,414.921826
std,10.878044,57.255843,7.443268,38.241255,2.812696,5.702835,42.391356,43.909309,47.886767,54.840617,...,200.292134,246.988560,290.578833,298.289660,299.680529,282.811489,237.267583,183.342283,142.148487,136.477327
min,17.000000,1.000000,2000.000000,0.000000,36.211400,-103.846600,0.000000,0.000000,0.000000,0.000000,...,84.043548,153.379333,263.736774,463.626667,612.769032,589.784516,388.617667,258.033548,186.626667,88.901667
25%,19.000000,45.000000,2005.000000,124.000000,39.059800,-96.177900,14.850000,17.707500,32.010000,57.330000,...,372.631452,585.783167,1008.830806,1548.044583,1802.153790,1684.873952,1285.802917,802.756048,494.868000,315.487581
50%,26.000000,91.000000,2011.000000,151.800000,40.988000,-91.479300,32.025000,36.840000,60.490000,87.870000,...,511.394677,767.745500,1211.288548,1746.882167,2003.823226,1859.227742,1439.530833,926.538226,594.222500,402.797419
75%,31.000000,139.000000,2018.000000,175.300000,43.345900,-86.446200,66.360000,66.080000,93.090000,121.957500,...,655.682581,934.913667,1405.645242,1936.222333,2206.897661,2068.388629,1595.977167,1052.381532,703.052333,505.692911
max,55.000000,239.000000,2025.000000,253.600000,48.814900,-80.748200,283.480000,343.800000,444.310000,493.550000,...,1306.850000,1517.362333,2163.205806,2737.994667,2947.784839,2712.901935,2299.902667,1532.883548,1123.550667,940.150323


In [6]:
yield_weather_df.describe(include='object')

,state_name,state_alpha,fips,county_name,short_desc,unit_desc,statisticcat_desc
count,24644,24644,24644,24644,24644,24644,24644
unique,13,13,1124,704,1,1,1
top,IOWA,IA,17011,JACKSON,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD
freq,2470,2470,26,231,24644,24644,24644


In [7]:
yield_weather_df.isnull().sum()

state_name        0
state_alpha       0
state_ansi        0
county_ansi       0
fips              0
                 ..
vp_mean_pa_m08    0
vp_mean_pa_m09    0
vp_mean_pa_m10    0
vp_mean_pa_m11    0
vp_mean_pa_m12    0
Length: 97, dtype: int64

In [8]:
yield_weather_df.duplicated().sum()

0

In [9]:
for i in yield_weather_df.select_dtypes(include='object').columns:
    print(yield_weather_df[i].value_counts())
    print("*"*30)

state_name
IOWA            2470
ILLINOIS        2417
INDIANA         2142
KENTUCKY        2092
KANSAS          2090
NEBRASKA        2069
OHIO            2051
MISSOURI        1967
MINNESOTA       1867
WISCONSIN       1577
MICHIGAN        1485
SOUTH DAKOTA    1363
NORTH DAKOTA    1054
Name: count, dtype: int64
******************************
state_alpha
IA    2470
IL    2417
IN    2142
KY    2092
KS    2090
NE    2069
OH    2051
MO    1967
MN    1867
WI    1577
MI    1485
SD    1363
ND    1054
Name: count, dtype: int64
******************************
fips
17011    26
27169    26
26161    26
26147    26
26115    26
         ..
29055     1
29119     1
29229     1
29209     1
21189     1
Name: count, Length: 1124, dtype: int64
******************************
county_name
JACKSON       231
WASHINGTON    217
JEFFERSON     199
CLAY          196
MONROE        184
             ... 
MAGOFFIN        1
STONE           1
DENT            1
OWSLEY          1
ELLIOTT         1
Name: count, Length: 704, dty

## Make one row per county

Soil data does not change between the years. So, to not query the soil data access as much, we will just do each county once

In [10]:
county_locations = yield_weather_df[["fips", "state_alpha", "county_name"]].drop_duplicates("fips").reset_index(drop=True)

In [11]:
county_locations.shape

(1124, 3)

In [12]:
county_locations.head()

,fips,state_alpha,county_name
0,17011,IL,BUREAU
1,17015,IL,CARROLL
2,17073,IL,HENRY
3,17085,IL,JO DAVIESS
4,17103,IL,LEE


## Get Soil Data

The soil data are downloaded from the USDA Soil Data Access service.

SSURGO organizes soil data into several levels:

1. A county contains multiple soil map units.
2. A map unit can contain multiple soil components.
3. A component can contain multiple soil layers, called horizons.

The query collects the map units, components, horizons, and soil measurements
needed to calculate representative county values.

The data are downloaded one county at a time. Each county is saved as a
separate CSV file so the download can be restarted without downloading every
county again.

In [13]:
SDA_URL = "https://SDMDataAccess.sc.egov.usda.gov/Tabular/post.rest"

In [14]:
# Make directory to save each of the soil files in
os.makedirs("../data/raw/soil_counties", exist_ok=True)

### Important Parts of the Soil Query

The query uses:

- `county_mapunit_acres` to measure how much of each map unit is inside the
  county.
- `comppct_r` to measure how common each soil component is within its map unit.
- `hzdept_r` and `hzdepb_r` to identify the top and bottom of each soil layer.
- Soil-property columns such as sand, clay, organic matter, pH, and bulk
  density.

Only soil horizons that overlap the upper 30 centimeters are selected.

In [15]:
for index, row in county_locations.iterrows():

    # The SDA uses the county intiial (Ex: IN) and the last 3 of fips code
    county_symbol = row["state_alpha"] + row["fips"][-3:]

    county_file = f"../data/raw/soil_counties/soil_{row["fips"]}.csv"

    if os.path.exists(county_file):
        print(f"{index}/{len(county_locations)}: {row["fips"]} already completed")
        continue

    print(f"Downloading county {index + 1} of {len(county_locations)}:", row["state_alpha"], row["fips"], county_symbol)

    query = f"""
    SELECT
        lo.areasymbol AS county_symbol,
        muao.mukey,
        muao.areaovacres AS county_mapunit_acres,
        mua.aws0100wta,
        mua.slopegradwta,
        co.comppct_r,
        ch.hzdept_r,
        ch.hzdepb_r,
        ch.sandtotal_r,
        ch.silttotal_r,
        ch.claytotal_r,
        ch.om_r,
        ch.ph1to1h2o_r,
        ch.dbthirdbar_r,
        ch.ksat_r
    
    FROM laoverlap AS lo
    
    INNER JOIN muaoverlap AS muao
        ON lo.lareaovkey = muao.lareaovkey
    
    INNER JOIN mapunit AS mu
        ON muao.mukey = mu.mukey
    
    LEFT JOIN muaggatt AS mua
        ON muao.mukey = mua.mukey
    
    INNER JOIN component AS co
        ON muao.mukey = co.mukey
    
    INNER JOIN chorizon AS ch
        ON co.cokey = ch.cokey
    
    WHERE
        lo.areatypename = 'County or Parish'
        AND lo.areasymbol = '{county_symbol}'
        AND ch.hzdept_r < 30
        AND ch.hzdepb_r > 0
    """

    parameters = {
        "query": query,
        "format": "JSON+COLUMNNAME"
    }

    try:

        response = requests.post(SDA_URL, json=parameters, timeout=120)

        response.raise_for_status()

        soil_json = response.json()

        columns = soil_json["Table"][0]
        rows = soil_json["Table"][1:]

        soil_data = pd.DataFrame(rows, columns=columns)

        soil_data["fips"] = row["fips"]

        soil_data.to_csv(county_file, index=False)

        print("Saved")

    except Exception as error:

        print("Failed:", row["fips"], error)

    time.sleep(0.1)

Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Saved
Save

## Combine all of the soil data files

In [16]:
temp = pd.read_csv('../data/raw/soil_counties/soil_17001.csv')

In [17]:
temp.head()

,county_symbol,mukey,county_mapunit_acres,aws0100wta,slopegradwta,comppct_r,hzdept_r,hzdepb_r,sandtotal_r,silttotal_r,claytotal_r,om_r,ph1to1h2o_r,dbthirdbar_r,ksat_r,fips
0,IL001,2378631,6345,22.94,7.5,90.0,0,18,4.0,73.0,23.0,2.50,6.2,1.45,9.17,17001
1,IL001,2378631,6345,22.94,7.5,90.0,18,94,4.0,65.0,31.0,0.25,5.8,1.50,9.17,17001
2,IL001,2378632,753,12.36,14.0,94.0,0,13,8.0,69.0,23.0,2.00,5.8,1.42,9.17,17001
3,IL001,2378632,753,12.36,14.0,94.0,13,75,8.0,58.0,34.0,0.25,5.0,1.47,0.91,17001
4,IL001,2378632,753,12.36,14.0,2.0,15,67,33.0,37.0,30.0,0.60,5.9,1.49,9.17,17001


In [18]:
all_soil_data = []

for filename in os.listdir("../data/raw/soil_counties"):
    if filename.endswith(".csv"):
        try:
            data = pd.read_csv(f"../data/raw/soil_counties/{filename}", dtype={"fips": str, "mukey":str})
            all_soil_data.append(data)
        except pd.errors.EmptyDataError:
            df = pd.DataFrame()  # Fallback to an empty DataFrame


In [19]:
soil_raw_df = pd.concat(all_soil_data, ignore_index=True)
soil_raw_df.head()

,county_symbol,mukey,county_mapunit_acres,aws0100wta,slopegradwta,comppct_r,hzdept_r,hzdepb_r,sandtotal_r,silttotal_r,claytotal_r,om_r,ph1to1h2o_r,dbthirdbar_r,ksat_r,fips
0,OH119,537849,1215.0,13.08,5.4,5.0,0,8,13.0,67.0,20.0,3.0,5.7,1.38,9.17,39119
1,OH119,537849,1215.0,13.08,5.4,5.0,15,28,7.0,67.0,26.0,1.0,5.2,1.37,9.17,39119
2,OH119,537849,1215.0,13.08,5.4,5.0,8,15,11.0,69.0,20.0,2.0,5.0,1.39,9.17,39119
3,OH119,537849,1215.0,13.08,5.4,5.0,28,81,15.0,52.0,33.0,0.5,5.1,1.50,2.70,39119
4,OH119,537849,1215.0,13.08,5.4,5.0,0,11,16.0,70.0,14.0,3.9,4.7,1.20,9.17,39119


In [20]:
soil_raw_df.tail()

,county_symbol,mukey,county_mapunit_acres,aws0100wta,slopegradwta,comppct_r,hzdept_r,hzdepb_r,sandtotal_r,silttotal_r,claytotal_r,om_r,ph1to1h2o_r,dbthirdbar_r,ksat_r,fips
715666,KY029,550964,873.0,11.85,4.0,5.0,20,71,7.0,67.0,26.0,0.75,5.5,1.42,9.00,21029
715667,KY029,550964,873.0,11.85,4.0,5.0,0,15,12.0,58.0,30.0,1.75,7.0,1.48,0.92,21029
715668,KY029,550964,873.0,11.85,4.0,5.0,15,46,10.0,42.0,48.0,0.25,8.2,1.53,0.92,21029
715669,KY029,550964,873.0,11.85,4.0,85.0,0,18,15.0,63.0,22.0,2.25,6.7,1.30,9.11,21029
715670,KY029,550964,873.0,11.85,4.0,85.0,18,74,7.0,48.0,45.0,0.30,5.8,1.43,0.55,21029


In [21]:
soil_raw_df.columns

Index(['county_symbol', 'mukey', 'county_mapunit_acres', 'aws0100wta',
       'slopegradwta', 'comppct_r', 'hzdept_r', 'hzdepb_r', 'sandtotal_r',
       'silttotal_r', 'claytotal_r', 'om_r', 'ph1to1h2o_r', 'dbthirdbar_r',
       'ksat_r', 'fips'],
      dtype='object')

In [22]:
soil_raw_df.shape

(715671, 16)

In [23]:
soil_raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 715671 entries, 0 to 715670
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   county_symbol         715671 non-null  object 
 1   mukey                 715671 non-null  object 
 2   county_mapunit_acres  715638 non-null  float64
 3   aws0100wta            715344 non-null  float64
 4   slopegradwta          715542 non-null  float64
 5   comppct_r             714271 non-null  float64
 6   hzdept_r              715671 non-null  int64  
 7   hzdepb_r              715671 non-null  int64  
 8   sandtotal_r           697583 non-null  float64
 9   silttotal_r           697225 non-null  float64
 10  claytotal_r           700662 non-null  float64
 11  om_r                  712197 non-null  float64
 12  ph1to1h2o_r           701376 non-null  float64
 13  dbthirdbar_r          711878 non-null  float64
 14  ksat_r                715399 non-null  float64
 15  

## Calculate Soil-Layer Thickness

Some soil layers extend below 30 cm. This project uses only the portion of each
layer that falls between 0 and 30 cm.

For example:

- A layer from 0–10 cm contributes 10 cm.
- A layer from 10–40 cm contributes 20 cm because only the 10–30 cm portion is
  used.
- A layer beginning below 30 cm is not included.

This makes sure the final values represent the upper 30 cm of soil.

In [24]:
soil_raw_df["top_cm"] = (soil_raw_df["hzdept_r"].clip(lower=0, upper=30))

In [25]:
soil_raw_df["bottom_cm"] = (soil_raw_df["hzdepb_r"].clip(lower=0,upper=30))

In [26]:
soil_raw_df["thickness_cm"] = (soil_raw_df["bottom_cm"] - soil_raw_df["top_cm"]).clip(lower=0)

## Calculate the Soil Weight

The soil rows should not all contribute equally. Some represent more land,
more common soil components, or thicker soil layers.

The weight is calculated using:

$$
\text{soil weight}
=
\text{county map-unit acres}
\times
\frac{\text{component percent}}{100}
\times
\text{layer thickness}
$$

This gives more influence to soil records that represent a larger portion of
the county's upper 30 cm of soil.

In [27]:
soil_raw_df["soil_weight"] = (soil_raw_df["county_mapunit_acres"] * (soil_raw_df["comppct_r"] / 100) * soil_raw_df["thickness_cm"])

## Soil Properties

The following soil properties are calculated for each county:

| Property | Simple meaning |
|---|---|
| Sand | Larger soil particles that usually allow faster drainage |
| Silt | Medium-sized soil particles that hold more water than sand |
| Clay | Very small soil particles that hold water and nutrients |
| Organic matter | Decomposed plant and animal material in the soil |
| pH | How acidic or alkaline the soil is |
| Bulk density | How compacted the soil is |
| Hydraulic conductivity | How easily water moves through saturated soil |
| Available water storage | How much water the soil can store for plants |
| Slope | How steep the land is |

These variables may help explain why corn yield and the effects of weather
differ between counties.

In [28]:
horizon_properties = {
    "sandtotal_r": "sand_pct_0_30cm",
    "silttotal_r": "silt_pct_0_30cm",
    "claytotal_r": "clay_pct_0_30cm",
    "om_r": "organic_matter_pct_0_30cm",
    "ph1to1h2o_r": "ph_0_30cm",
    "dbthirdbar_r": "bulk_density_g_cm3_0_30cm",
    "ksat_r": "ksat_um_s_0_30cm"
}

## Calculate One Soil Row per County

For sand, silt, clay, organic matter, pH, bulk density, and hydraulic
conductivity, the county average is calculated as:

$$
\text{county average}
=
\frac{
\sum(\text{soil value} \times \text{soil weight})
}{
\sum(\text{soil weight})
}
$$

Rows missing a particular property are ignored only for that property.

Available water storage and slope are already summarized by NRCS at the map-unit
level. They are therefore weighted only by the number of map-unit acres inside
the county.

In [29]:
county_soil_results = []

for fips, county_data in soil_raw_df.groupby("fips"):
    county_result = {"fips": fips}

    for original_column, final_column in horizon_properties.items():
        usable_data = county_data.dropna(subset=[original_column, "soil_weight"])
        total_weight = usable_data["soil_weight"].sum()

        if total_weight > 0:
            weighted_value = (usable_data[original_column] * usable_data["soil_weight"]).sum() / total_weight

        else:
            weighted_value = np.nan

        county_result[final_column] = weighted_value

    
    mapunit_data = (county_data.drop_duplicates("mukey").copy())

    usable_aws = mapunit_data.dropna(subset=["aws0100wta", "county_mapunit_acres"])

    if len(usable_aws) > 0:
        county_result["available_water_storage_cm_0_100cm"] = np.average(usable_aws["aws0100wta"], weights=usable_aws["county_mapunit_acres"])

    else:
        county_result["available_water_storage_cm_0_100cm"] = np.nan

    usable_slope = mapunit_data.dropna(subset=["slopegradwta", "county_mapunit_acres"])

    if len(usable_slope) > 0:
        county_result["slope_pct"] = np.average(usable_slope["slopegradwta"], weights=usable_slope["county_mapunit_acres"])

    else:
        county_result["slope_pct"] = np.nan

    county_soil_results.append(county_result)

## Create final soil Dataframe

In [30]:
soil_df = pd.DataFrame(county_soil_results)

In [31]:
soil_df.to_csv('../data/raw/corn_belt_soil.csv', index=False)

In [32]:
soil_df.head()

,fips,sand_pct_0_30cm,silt_pct_0_30cm,clay_pct_0_30cm,organic_matter_pct_0_30cm,ph_0_30cm,bulk_density_g_cm3_0_30cm,ksat_um_s_0_30cm,available_water_storage_cm_0_100cm,slope_pct
0,17001,9.761685,66.093087,24.145839,1.965400,6.272974,1.392362,7.827496,18.455786,6.672519
1,17003,14.200159,60.170257,25.629584,1.916471,6.198012,1.372761,10.448577,17.100846,9.697656
2,17005,12.001764,67.758068,20.240168,1.713029,6.137333,1.405465,7.060548,17.364135,4.503059
3,17007,14.919208,62.206710,22.800581,3.879057,6.458088,1.350217,11.356888,18.413318,2.278609
4,17009,10.585626,66.772594,22.641780,1.654722,6.189540,1.411360,8.372990,18.366319,13.094994


---

In [33]:
soil_df.shape

(1124, 10)

## Merge with Corn Yield and Weather Data

In [34]:
yield_weather_df["fips"] = (yield_weather_df["fips"].astype(str).str.zfill(5))

soil_df["fips"] = (soil_df["fips"].astype(str).str.zfill(5))

grand_final_df = yield_weather_df.merge(soil_df, on="fips", how="left", validate="many_to_one")

In [35]:
grand_final_df.head()

,state_name,state_alpha,state_ansi,county_ansi,fips,county_name,year,short_desc,unit_desc,statisticcat_desc,...,vp_mean_pa_m12,sand_pct_0_30cm,silt_pct_0_30cm,clay_pct_0_30cm,organic_matter_pct_0_30cm,ph_0_30cm,bulk_density_g_cm3_0_30cm,ksat_um_s_0_30cm,available_water_storage_cm_0_100cm,slope_pct
0,ILLINOIS,IL,17,11,17011,BUREAU,2025,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,...,357.549355,12.944472,62.969536,23.948762,3.691334,6.516119,1.363566,11.164841,18.336345,4.666624
1,ILLINOIS,IL,17,11,17011,BUREAU,2024,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,...,462.975000,12.944472,62.969536,23.948762,3.691334,6.516119,1.363566,11.164841,18.336345,4.666624
2,ILLINOIS,IL,17,11,17011,BUREAU,2023,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,...,614.017097,12.944472,62.969536,23.948762,3.691334,6.516119,1.363566,11.164841,18.336345,4.666624
3,ILLINOIS,IL,17,11,17011,BUREAU,2022,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,...,393.786774,12.944472,62.969536,23.948762,3.691334,6.516119,1.363566,11.164841,18.336345,4.666624
4,ILLINOIS,IL,17,11,17011,BUREAU,2021,"CORN, GRAIN - YIELD, MEASURED IN BU / ACRE",BU / ACRE,YIELD,...,504.136452,12.944472,62.969536,23.948762,3.691334,6.516119,1.363566,11.164841,18.336345,4.666624


## Save to CSV

In [36]:
grand_final_df.to_csv('../data/processed/corn_belt_yield_weather_soil.csv', index=False)